# Setup

In [40]:
"""
หากยังไม่มี library ใด ให้ติดตั้งก่อน
!pip install -q nltk wordcloud scikit-learn pandas matplotlib tqdm huggingface_hub
!pip install -q pythainlp          # ใช้เมื่อทำงานกับข้อความภาษาไทย
"""

import json, os, re, random, itertools, time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
for pkg in ["punkt_tab", "stopwords", "wordnet", "vader_lexicon"]:
    nltk.download(pkg, quiet=True)

%matplotlib inline

RANDOM_SEED = 42
SAMPLE_N = 30000         # เวอร์ชันลด workload: Part 2 ใช้อย่างน้อย 20,000 รีวิว
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)



# Load

In [41]:
# ดาวน์โหลดไฟล์เฉพาะหมวดที่เลือก
from huggingface_hub import hf_hub_download
"""
CATEGORY = "Office_Products"   # <-- เปลี่ยนเป็นหมวดที่ทีมจอง

review_path = hf_hub_download(repo_id="McAuley-Lab/Amazon-Reviews-2023",
                              filename=f"raw/review_categories/{CATEGORY}.jsonl",
                              repo_type="dataset", local_dir=r"W:\Learn_Program\project_bigdata\Part_2\data")
meta_path   = hf_hub_download(repo_id="McAuley-Lab/Amazon-Reviews-2023",
                              filename=f"raw/meta_categories/meta_{CATEGORY}.jsonl",
                              repo_type="dataset", local_dir=r"W:\Learn_Program\project_bigdata\Part_2\data")
print(review_path, meta_path)
"""

<>:8: SyntaxWarning: invalid escape sequence '\L'
<>:8: SyntaxWarning: invalid escape sequence '\L'
C:\Users\thana\AppData\Local\Temp\ipykernel_33484\2109258803.py:8: SyntaxWarning: invalid escape sequence '\L'
  repo_type="dataset", local_dir=r"W:\Learn_Program\project_bigdata\Part_2\data")


'\nCATEGORY = "Office_Products"   # <-- เปลี่ยนเป็นหมวดที่ทีมจอง\n\nreview_path = hf_hub_download(repo_id="McAuley-Lab/Amazon-Reviews-2023",\n                              filename=f"raw/review_categories/{CATEGORY}.jsonl",\n                              repo_type="dataset", local_dir=r"W:\\Learn_Program\\project_bigdata\\Part_2\\data")\nmeta_path   = hf_hub_download(repo_id="McAuley-Lab/Amazon-Reviews-2023",\n                              filename=f"raw/meta_categories/meta_{CATEGORY}.jsonl",\n                              repo_type="dataset", local_dir=r"W:\\Learn_Program\\project_bigdata\\Part_2\\data")\nprint(review_path, meta_path)\n'

In [42]:
review_path = r"W:\Learn_Program\project_bigdata\Part_2\data\raw\Office_Products.jsonl"
meta_path = r"W:\Learn_Program\project_bigdata\Part_2\data\raw\meta_Office_Products.jsonl"

## Load review

In [43]:
import json
import random
import pandas as pd

# กำหนดจำนวนตัวอย่างที่ต้องการสุ่ม และล็อค Seed เพื่อให้ผลการสุ่มคงที่
N = SAMPLE_N
random.seed(RANDOM_SEED)

samples = []      # ลิสต์สำหรับเก็บข้อมูลตัวอย่าง (Reservoir)
valid_count = 0  # นับจำนวนบรรทัดที่แปลงเป็น JSON สำเร็จ (ใช้คำนวณโอกาสสุ่ม)

# เปิดไฟล์อ่านทีละบรรทัดเพื่อประหยัดหน่วยความจำ (RAM)
with open(review_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            # แปลงข้อความ JSON ในบรรทัดนั้นให้เป็น Dictionary
            record = json.loads(line)
            valid_count += 1  # นำนับเฉพาะบรรทัดที่อ่านได้สำเร็จ
            
            # --- Reservoir Sampling Algorithm ---
            # 1. หากเก็บข้อมูลยังไม่ครบ N ให้ใส่ลงในลิสต์ทันที
            if len(samples) < N:
                samples.append(record)
            else:
                # 2. หากเกิน N ให้สุ่มตำแหน่ง j ตั้งแต่ 0 ถึง valid_count - 1
                j = random.randint(0, valid_count - 1)
                
                # ถ้าค่าที่สุ่มได้ตกอยู่ในช่วง 0 ถึง N - 1 ให้สุ่มแทนที่ข้อมูลเก่าในตำแหน่งนั้น
                if j < N:
                    samples[j] = record
                    
        except json.JSONDecodeError:
            # ข้ามบรรทัดที่รูปแบบ JSON เสีย/ไม่ถูกต้อง โดยไม่นำมาคำนวณ
            continue

# แปลงลิสต์ตัวอย่างที่สุ่มได้ให้กลายเป็น Pandas DataFrame
df_review = pd.DataFrame(samples)

# แสดงจำนวนรายการทั้งหมดที่สุ่มมาได้
print("จำนวน Review ที่สุ่มได้:", len(df_review))

จำนวน Review ที่สุ่มได้: 30000


### ผลของการสุ่ม

In [44]:
print(df_review.shape)

display(df_review.head())

print(df_review.columns.tolist())

(30000, 10)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3.0,Doesn't stick to the desk or keyboard mount,"Not super comfy, but I guess can't expect much...",[],B01M11FLUJ,B0C3VN2M6Z,AF4DPMP75Z5ZZOLRSE76HF5C6V2Q,1613179251487,0,True
1,5.0,"I should have purchased this years ago, Finall...","I should have purchased this years ago, Final...",[],B00DRIPNTA,B00DRIPNTA,AFLJW74BMJAKJWBXCCHH4RHFF5VQ,1419306054000,1,True
2,5.0,Great Jack Russell Calendar,Jack Russell Terriers are one of our favorite ...,[],B00DYQJXR8,B00DYQJXR8,AHIDQ2EZX6PC4LXCSCBF22I37FYQ,1389460846000,0,True
3,5.0,Precise and high quality color,This is the perfect set of pens. The colors ar...,[],B073GNFTMB,B08HZ7CVD6,AEXC5UUSS65DQ6IPIZLYOPBKNERQ,1638838712173,0,True
4,5.0,Five Stars,Works fine. No issues.,[],B001FNRLHQ,B00ZLLMAYW,AEXPINSL73TBVELTAE7FJ2LFP5LQ,1461331431000,0,True


['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


### ตรวจสอบว่ามี รีวิวที่ซ้ำกันมั้ย

In [45]:
review_key = [
    "asin",
    "user_id",
    "timestamp",
    "text"
]

duplicate_count = df_review.duplicated(
    subset=review_key
).sum()

print("Duplicate reviews:", duplicate_count)

Duplicate reviews: 2


In [57]:
duplicates = df_review[
    df_review.duplicated(
        subset=review_key,
        keep=False
    )
]

display(duplicates.head(20))

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
2158,5.0,A Great Purchase for an Organized Life,I recently purchased the adult chore board rem...,[{'small_image_url': 'https://m.media-amazon.c...,B0B65CR97S,B0B65CR97S,AHJTH7EEWTXZWLG6ON4A455E5HQQ,1673940539490,4,True
6904,5.0,I use this scale in a restaurant setting easil...,I use this scale in a restaurant setting easil...,[],B00IZ1YHZK,B00IZ1YHZK,AHCHDRED5AQF3BAVVCXPL3W6WI4Q,1454035454000,0,True
7842,5.0,I use this scale in a restaurant setting easil...,I use this scale in a restaurant setting easil...,[],B00IZ1YHZK,B00IZ1YHZK,AHCHDRED5AQF3BAVVCXPL3W6WI4Q,1454035454000,0,True
18973,5.0,A Great Purchase for an Organized Life,I recently purchased the adult chore board rem...,[{'small_image_url': 'https://m.media-amazon.c...,B0B65CR97S,B0B65CR97S,AHJTH7EEWTXZWLG6ON4A455E5HQQ,1673940539490,4,True


## load meta Product

### ดูว่าจะต้องดึงข้อมูล Product อันไหน

In [47]:
needed_parent_asin = set(
    df_review["parent_asin"]
    .dropna()
    .astype(str)
)

print(
    "จำนวน parent_asin ที่ต้องหา:",
    len(needed_parent_asin)
)

จำนวน parent_asin ที่ต้องหา: 20933


### ดึงข้อมูล Product

In [48]:
product_records = []
bad_meta_lines = 0

with open(meta_path, "r", encoding="utf-8") as f:

    for line_number, line in enumerate(f, start=1):

        try:
            product = json.loads(line)

            parent_asin = product.get("parent_asin")

            if parent_asin is not None:
                parent_asin = str(parent_asin)

            if parent_asin in needed_parent_asin:
                product_records.append(product)

        except json.JSONDecodeError:
            bad_meta_lines += 1
            continue

print("Products found:", len(product_records))
print("Bad metadata lines:", bad_meta_lines)

Products found: 20933
Bad metadata lines: 0


### ดูข้อมูล Product ว่าเป็นยังไง

In [49]:
df_product = pd.DataFrame(product_records)

print(df_product.shape)

display(df_product.head())

(20933, 16)


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Office Products,"Avery Printable Blank Square Labels, 2"" x 2"", ...",4.8,4541,[Create professional looking labels with matte...,"[A simple way to display your creative side, t...",8.13,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Avery Sure Feed Labels (Small Busi...,AVERY,"[Office Products, Office & School Supplies, La...","{'Manufacturer': 'Avery', 'Brand': 'AVERY', 'I...",B0B6QVDF6Q,None,NaN,NaN
1,All Electronics,"Bonsaii 12-Sheet Cross Cut Paper Shredder, 5.5...",4.5,475,[【P-4 Level Security】Crosscut shredder for hom...,[],53.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Product Review of Bonsaii Paper Sh...,bonsaii,"[Office Products, Office Electronics, Other Of...",{'Product Dimensions': '12.8 x 7.68 x 15.75 in...,B0B1BVCJLP,None,NaN,NaN
2,Amazon Home,Comfy Mee Coccyx Orthopedic Memory Foam Office...,3.9,35,[],[],None,[{'thumb': 'https://m.media-amazon.com/images/...,[],NaN,"[Office Products, Office Furniture & Lighting,...",{'Package Dimensions': '17.8 x 13.9 x 2.99 inc...,B06XD2FCT9,None,NaN,NaN
3,Office Products,"Amazon Basics Multipurpose Copy Printer Paper,...",4.8,135510,[1 ream (500 sheets) of 8.5 x 11 white copier ...,"[Product Description, Amazon Basics Multipurpo...",8.49,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Amazon Basics Multipurpose Copy Pa...,Amazon Basics,"[Office Products, Office & School Supplies, Pa...","{'Manufacturer': 'Amazon Basics', 'Brand': 'Am...",B0C36Q7PM4,None,NaN,NaN
4,NaN,JuneLsy Business Card Holder Case - Profession...,4.8,5558,[MATERIAL: Business Card Holder is made of hig...,"[Junelsy leather business card holder case, - ...",6.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Cute compact business card holder'...,JuneLsy,"[Office Products, Office & School Supplies, De...","{'Manufacturer': 'JuneLsy', 'Brand': 'JuneLsy'...",B07R51VM88,None,NaN,NaN


### ตรวจสอบว่ามี Product ที่ซ้ำรึป่าว

In [50]:
duplicate_products = df_product[
    df_product.duplicated(
        subset="parent_asin",
        keep=False
    )
]

print(
    "Duplicate parent_asin:",
    duplicate_products["parent_asin"].nunique()
)

Duplicate parent_asin: 0


แก้ข้อมูลที่ซ้ำ

In [51]:
df_product_unique = (
    df_product
    .drop_duplicates(
        subset="parent_asin",
        keep="first"
    )
    .copy()
)

print(
    "Products before:",
    len(df_product)
)

print(
    "Products after:",
    len(df_product_unique)
)

Products before: 20933
Products after: 20933


## Merge review and product

In [52]:
df_merged = df_review.merge(
    df_product_unique,
    on="parent_asin",
    how="left",
    suffixes=("_review", "_product"),
    indicator=True,
    validate="many_to_one"
)

### ตรวจสอบว่า merge ได้กี่ %

In [53]:
total = len(df_merged)

matched = (
    df_merged["_merge"] == "both"
).sum()

unmatched = (
    df_merged["_merge"] == "left_only"
).sum()

print(f"Total Reviews    : {total:,}")
print(f"Joined           : {matched:,}")
print(f"Not Joined       : {unmatched:,}")
print(f"Join Success     : {matched / total * 100:.2f}%")
print(f"Missing Metadata : {unmatched / total * 100:.2f}%")
print("Before Join:", len(df_review))
print("After Join :", len(df_merged))
if len(df_merged) == len(df_review):
    print("✓ จำนวนแถวไม่เพิ่มหลัง Join")
else:
    print("⚠ จำนวนแถวเพิ่มหลัง Join")
    print("เพิ่มขึ้น:", len(df_merged) - len(df_review))

Total Reviews    : 30,000
Joined           : 30,000
Not Joined       : 0
Join Success     : 100.00%
Missing Metadata : 0.00%
Before Join: 30000
After Join : 30000
✓ จำนวนแถวไม่เพิ่มหลัง Join


### ตรวจสอบ Missing 

In [54]:
important_columns = [
    "parent_asin",
    "title",
    "price",
    "categories"
]

existing_columns = [
    col for col in important_columns
    if col in df_merged.columns
]

missing_summary = pd.DataFrame({
    "Missing": df_merged[existing_columns].isna().sum(),
    "Missing_%": (
        df_merged[existing_columns]
        .isna()
        .mean() * 100
    ).round(2)
})

display(missing_summary)

,Missing,Missing_%
parent_asin,0,0.00
price,8723,29.08
categories,0,0.00


# สรุป Data

In [55]:
duplicate_review_count = df_review.duplicated(
    subset=[
        "asin",
        "user_id",
        "timestamp",
        "text"
    ]
).sum()

duplicate_parent_count = df_product.duplicated(
    subset="parent_asin"
).sum()

missing_parent_review = df_review["parent_asin"].isna().sum()

missing_parent_product = df_product["parent_asin"].isna().sum()

print("=" * 50)
print("DATA QUALITY SUMMARY")
print("=" * 50)

print(f"Reviews sampled           : {len(df_review):,}")
print(f"Unique parent_asin        : {df_review['parent_asin'].nunique():,}")
print(f"Duplicate reviews         : {duplicate_review_count:,}")
print(f"Missing review parent_asin: {missing_parent_review:,}")

print()

print(f"Products found            : {len(df_product):,}")
print(f"Duplicate product rows    : {duplicate_parent_count:,}")
print(f"Missing product parent_asin: {missing_parent_product:,}")

print()

print(f"Joined successfully       : {matched:,}")
print(f"Not joined               : {unmatched:,}")
print(f"Join success rate        : {matched / total * 100:.2f}%")

DATA QUALITY SUMMARY
Reviews sampled           : 30,000
Unique parent_asin        : 20,933
Duplicate reviews         : 2
Missing review parent_asin: 0

Products found            : 20,933
Duplicate product rows    : 0
Missing product parent_asin: 0

Joined successfully       : 30,000
Not joined               : 0
Join success rate        : 100.00%


# Representativeness

ข้อมูลชุดนี้เป็นตัวแทนของลูกค้าที่มีรีวิวสินค้าในหมวด Office Products
และมีข้อมูลอยู่ในชุด Amazon Reviews 2023 ที่นำมาสุ่มวิเคราะห์
จึงเหมาะสำหรับศึกษาความคิดเห็น ปัญหา และประสบการณ์ที่ลูกค้าสะท้อนผ่านรีวิว

อย่างไรก็ตาม ข้อมูลนี้ไม่เป็นตัวแทนของลูกค้า Amazon ทั้งหมด
เนื่องจากไม่ครอบคลุมลูกค้าที่ซื้อสินค้าแต่ไม่ได้เขียนรีวิว
รวมถึงลูกค้าที่ไม่ได้อยู่ในชุดข้อมูลที่นำมาวิเคราะห์ Representativeness

In [59]:
df_merged.to_csv(r"W:\Learn_Program\project_bigdata\Part_2\data\data.csv")